# Assignment 2 — LangGraph Workflow

Builds the `StateGraph`, renders it, and runs it on a retrieval question and a direct question to show the conditional edge, groundedness validation, and final report.

**Prerequisite:** a populated `.env` with valid Azure OpenAI credentials.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Build and visualize the graph

In [ ]:
from IPython.display import Image, display
from src.graph import build_graph

graph = build_graph()

# ASCII fallback always works; PNG needs a mermaid render backend.
print(graph.get_graph().draw_ascii())
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"(PNG render skipped: {exc})")

## 2. Run a RETRIEVAL question (routes through retrieve_context)

In [ ]:
from src.graph import run

state = run("My computer is infected with malware. What steps should I take?", thread_id="nb-g1")
print("Route:", state["request_type"])
print(state["final_report"])

## 3. Run a DIRECT question (skips retrieval via the conditional edge)

In [ ]:
state2 = run("Hello, what can you help me with?", thread_id="nb-g2")
print("Route:", state2["request_type"])
print(state2["final_report"])

## 4. Export the graph image and save a sample result

In [ ]:
import json
from src.graph import export_graph_png
from src.config import OUTPUTS_DIR

try:
    print("Graph PNG:", export_graph_png())
except Exception as exc:
    print(f"(PNG export skipped: {exc})")

sample = {
    "question": state["question"],
    "route": state["request_type"],
    "answer": state["answer"],
    "finding": state["finding"],
    "validation_result": state["validation_result"],
}
out = OUTPUTS_DIR / "sample_result.json"
out.write_text(json.dumps(sample, indent=2), encoding="utf-8")
print("Saved:", out)